# Built From Scratch



In [10]:
!pip install -q wandb

In [11]:
import os
import numpy as np
import pandas as pd
import wandb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score


INPUT_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "smart-mcq-solver-challenge"
    if not os.path.exists(INPUT_DIR):
        INPUT_DIR = "."

train_path = os.path.join(INPUT_DIR, "train.csv")
test_path  = os.path.join(INPUT_DIR, "test.csv")

print(f"Data Directory: {INPUT_DIR}")
print(f"Train path: {train_path} (exists: {os.path.exists(train_path)})")
print(f"Test path: {test_path} (exists: {os.path.exists(test_path)})")

Data Directory: /kaggle/input/competitions/smart-mcq-solver-challenge
Train path: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv (exists: True)
Test path: /kaggle/input/competitions/smart-mcq-solver-challenge/test.csv (exists: True)


## Feature Extraction & Helper Functions

In [12]:
def fit_tfidf(df):
    """
    Fits a TF-IDF vectorizer on all prompts and options.
    """
    all_texts = df['prompt'].tolist()
    for col in ['A', 'B', 'C', 'D', 'E']:
        if col in df.columns:
            all_texts.extend(df[col].dropna().astype(str).tolist())
            
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    vectorizer.fit(all_texts)
    return vectorizer

def extract_features(df, vectorizer, is_train=True):
    """
    Extracts features for each option of each question.
    Returns X (features) and y (binary labels if is_train=True).
    """
    rows_features = []
    labels = []
    
    option_cols = ['A', 'B', 'C', 'D', 'E']
    
    for idx, row in df.iterrows():
        prompt       = str(row['prompt'])
        prompt_words = set(prompt.lower().split())
        prompt_vec   = vectorizer.transform([prompt])
        
        option_texts = [str(row[col]) for col in option_cols]
        option_vecs  = vectorizer.transform(option_texts)
        
        # Calculate TF-IDF similarities between prompt and all options
        sims    = cosine_similarity(prompt_vec, option_vecs)[0]
        lengths = [len(t) for t in option_texts]
        avg_len = np.mean(lengths)
        max_len = np.max(lengths)
        
        for i, col in enumerate(option_cols):
            opt_text  = option_texts[i]
            opt_words = set(opt_text.lower().split())
            
            tfidf_sim  = sims[i]
            opt_len    = lengths[i]
            rel_len    = opt_len / (avg_len + 1e-5)
            is_longest = 1.0 if opt_len == max_len else 0.0
            overlap    = len(prompt_words.intersection(opt_words))
            jaccard    = len(prompt_words.intersection(opt_words)) / (len(prompt_words.union(opt_words)) + 1e-5)
            
            features = [tfidf_sim, opt_len, rel_len, is_longest, overlap, jaccard]
            rows_features.append(features)
            
            if is_train:
                is_correct = 1.0 if str(row['answer']) == col else 0.0
                labels.append(is_correct)
                
    X = np.array(rows_features)
    y = np.array(labels) if is_train else None
    return X, y

def map3_score(preds, targets):
    """
    Computes Mean Average Precision at 3 (MAP@3).
    """
    score = 0.0
    for pred, target in zip(preds, targets):
        for i, p in enumerate(pred):
            if p == target:
                score += 1.0 / (i + 1)
                break
    return score / len(targets)

## Load Data & Extract Features

In [13]:
df_train = pd.read_csv(train_path)
print(f"Loaded {len(df_train)} training examples.")

print("Fitting TF-IDF vectorizer...")
vectorizer = fit_tfidf(df_train)

print("Extracting features...")
X, y = extract_features(df_train, vectorizer, is_train=True)
print(f"X shape: {X.shape}, y shape: {y.shape}")


wandb.login()

run = wandb.init(
    project='smart-mcq-solver',
    name='tfidf-histgbm-scratch',
    config={
        'model'          : 'HistGradientBoostingClassifier',
        'model_type'     : 'built-from-scratch',
        'n_folds'        : 5,
        'max_iter'       : 100,
        'random_state'   : 42,
        'tfidf_ngram'    : '(1,2)',
        'n_features'     : X.shape[1],
        'train_size'     : len(df_train),
    }
)
print(f'WandB run initialized: {run.name}  |  URL: {run.url}')

Loaded 2000 training examples.
Fitting TF-IDF vectorizer...
Extracting features...
X shape: (10000, 6), y shape: (10000,)


WandB run initialized: tfidf-histgbm-scratch  |  URL: https://wandb.ai/vinodparvathy-indian-institute-of-technology-madras/smart-mcq-solver/runs/90x319k5


## 5-Fold Cross-Validation Evaluation

In [14]:
kf         = KFold(n_splits=5, shuffle=True, random_state=42)
map3_scores = []
acc_scores  = []
f1_scores   = []

option_cols = ['A', 'B', 'C', 'D', 'E']

print("--- Starting 5-Fold Cross-Validation ---")
for fold, (train_idx_groups, val_idx_groups) in enumerate(kf.split(df_train)):
    # Map question-level groups back to option-level indices
    train_idx = []
    for g in train_idx_groups:
        train_idx.extend(range(5 * g, 5 * g + 5))
        
    val_idx = []
    for g in val_idx_groups:
        val_idx.extend(range(5 * g, 5 * g + 5))
        
    X_train, y_train = X[train_idx], y[train_idx]
    X_val,   y_val   = X[val_idx],   y[val_idx]
    
    # Train model from scratch
    model = HistGradientBoostingClassifier(random_state=42, max_iter=100)
    model.fit(X_train, y_train)
    
    # Predict probability of being correct for validation options
    probs = model.predict_proba(X_val)[:, 1]
    
    # Predict rankings and compute MAP@3
    predictions = []
    val_targets = []
    
    for idx_in_val_groups, g in enumerate(val_idx_groups):
        g_probs        = probs[idx_in_val_groups * 5 : (idx_in_val_groups + 1) * 5]
        ranked_indices = np.argsort(g_probs)[::-1]
        top_3          = [option_cols[i] for i in ranked_indices[:3]]
        predictions.append(top_3)
        val_targets.append(df_train.iloc[g]['answer'])
        
    # Top-1 predictions for accuracy and F1
    top1_preds = [p[0] for p in predictions]
    
    score    = map3_score(predictions, val_targets)
    fold_acc = accuracy_score(val_targets, top1_preds)
    fold_f1  = f1_score(val_targets, top1_preds, average='macro', zero_division=0)
    
    map3_scores.append(score)
    acc_scores.append(fold_acc)
    f1_scores.append(fold_f1)
    
    print(f"Fold {fold+1}  MAP@3: {score:.5f}  |  Accuracy: {fold_acc:.5f}  |  F1: {fold_f1:.5f}")
    
    # ── Log per-fold metrics to WandB ──────────────────────────────────────
    wandb.log({
        'fold'          : fold + 1,
        'fold_map@3'    : score,
        'fold_accuracy' : fold_acc,
        'fold_f1_macro' : fold_f1,
    })

mean_map3 = np.mean(map3_scores)
mean_acc  = np.mean(acc_scores)
mean_f1   = np.mean(f1_scores)
print(f"\nMean  MAP@3: {mean_map3:.5f}  |  Accuracy: {mean_acc:.5f}  |  F1: {mean_f1:.5f}")

# ── Log final summary to WandB ─────────────────────────────────────────────
wandb.summary['mean_map@3']      = mean_map3
wandb.summary['mean_accuracy']   = mean_acc
wandb.summary['mean_f1_macro']   = mean_f1
wandb.summary['best_fold_map@3'] = max(map3_scores)

--- Starting 5-Fold Cross-Validation ---
Fold 1  MAP@3: 0.94375  |  Accuracy: 0.90500  |  F1: 0.90653
Fold 2  MAP@3: 0.91625  |  Accuracy: 0.86000  |  F1: 0.86237
Fold 3  MAP@3: 0.92250  |  Accuracy: 0.86500  |  F1: 0.86513
Fold 4  MAP@3: 0.92417  |  Accuracy: 0.87000  |  F1: 0.86969
Fold 5  MAP@3: 0.91417  |  Accuracy: 0.85000  |  F1: 0.84907

Mean  MAP@3: 0.92417  |  Accuracy: 0.87000  |  F1: 0.87056


## Train Final Model on Complete Dataset

In [15]:
print("Training final model on the full training set...")
final_model = HistGradientBoostingClassifier(random_state=42, max_iter=100)
final_model.fit(X, y)
print("Training complete!")

Training final model on the full training set...
Training complete!


## Generate Submission File on Test Set

In [16]:
df_test = pd.read_csv(test_path)
print(f"Loaded {len(df_test)} test examples.")

print("Extracting features from test set...")
X_test, _ = extract_features(df_test, vectorizer, is_train=False)

print("Generating test predictions...")
probs_test       = final_model.predict_proba(X_test)[:, 1]
predictions_test = []
option_cols      = ['A', 'B', 'C', 'D', 'E']

for g in range(len(df_test)):
    g_probs        = probs_test[g * 5 : (g + 1) * 5]
    ranked_indices = np.argsort(g_probs)[::-1]
    top_3          = [option_cols[i] for i in ranked_indices[:3]]
    predictions_test.append(" ".join(top_3))

submission_df = pd.DataFrame({
    'ID'        : df_test['id'],
    'Prediction': predictions_test
})
submission_df.to_csv("submission.csv", index=False)
print("Submission file 'submission.csv' generated successfully!")

# ── Log submission artifact to WandB and finish run ────────────────────────
artifact = wandb.Artifact('submission-scratch', type='predictions')
artifact.add_file('submission.csv')
wandb.log_artifact(artifact)
wandb.finish()
print('WandB run finished! View at https://wandb.ai')

submission_df.head(10)

Loaded 500 test examples.
Extracting features from test set...
Generating test predictions...
Submission file 'submission.csv' generated successfully!


fold,▁▃▅▆█
fold_accuracy,█▂▃▄▁
fold_f1_macro,█▃▃▄▁
fold_map@3,█▁▃▃▁
best_fold_map@3,0.94375
fold,5
fold_accuracy,0.85
fold_f1_macro,0.84907
fold_map@3,0.91417
mean_accuracy,0.87
mean_f1_macro,0.87056


WandB run finished! View at https://wandb.ai


,ID,Prediction
0,1,A C B
1,2,B E D
2,3,B E C
3,4,E C D
4,5,C E B
5,6,C D B
6,7,E B D
7,8,B D A
8,9,D C B
9,10,E B C
